# 03 - Blob Storage indexer and enrichment skills

Build the pipeline explicitly: **Blob Storage -> data source -> indexer -> Text Split + Shaper skills -> chunk index -> keyword search**. Nothing uploads documents directly into Search: the indexer must read the blobs and execute the skillset.

Text Split creates overlapping text chunks; Shaper packages source metadata into a provenance object. These are non-billable utility skills. There is no Image Analysis, Azure Vision embedding, OCR, embedding model, or additional Foundry deployment in this lab. Scanned PDFs and image-only files need a separate extraction lab.

Use the deployed Central US Search and Storage resources. Your user needs Search Service Contributor and Search Index Data Contributor on Search, and Storage Blob Data Contributor on the workshop container. Search's system-assigned managed identity needs Storage Blob Data Reader on that container; the Bicep deployment already grants it. User credentials upload the source files; Search's identity reads them.

Use synthetic, non-sensitive data only. Each attendee chooses a unique prefix. Prefixes avoid collisions, not access control: attendees share the Search service and container.

## 1. Configure your lab

Run `az login --tenant <workshop-tenant-id>` in PowerShell, select the workshop kernel, and run this notebook from the workshop root or `notebooks` directory. Optionally set `WORKSHOP_PREFIX` in your untracked `.env` to a unique 3-20 character value using lowercase letters and digits, for example `alice01`. Restart the kernel after changing environment files.

Every REST request is visible. Token providers refresh credentials without hiding the request, URL, headers or JSON payload.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import quote

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_STORAGE_RESOURCE_ID',
            'AZURE_STORAGE_BLOB_ENDPOINT', 'AZURE_STORAGE_CONTAINER_NAME')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Run the deployment first.')
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
storage_endpoint = os.environ['AZURE_STORAGE_BLOB_ENDPOINT'].rstrip('/')
storage_id = os.environ['AZURE_STORAGE_RESOURCE_ID']
container_name = os.environ['AZURE_STORAGE_CONTAINER_NAME']
api_version = '2025-09-01'
index_name = f'{prefix}-blob-chunks'
data_source_name = f'{prefix}-blob-source'
skillset_name = f'{prefix}-blob-skills'
indexer_name = f'{prefix}-blob-indexer'
blob_prefix = f'{prefix}/blob/'
credential = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
storage_token = get_bearer_token_provider(credential, 'https://storage.azure.com/.default')
print('Search:', search_endpoint)
print('Index:', index_name)
print('Source folder:', f'{container_name}/{blob_prefix}')


## 2. Upload source documents to Blob Storage

The repository includes two original synthetic text documents. Upload only these files to your prefix; repeated uploads overwrite only the same two blobs. Add your own text-based PDF, DOCX, PPTX, HTML or TXT files to the same folder later using Storage Explorer. This lab's parser extracts text, not image content.

The storage endpoint remains private to authenticated callers (not anonymously readable). No account key or SAS is used.

In [ ]:
sample_files = sorted((root / 'data' / 'search').glob('*.txt'))
if len(sample_files) != 2:
    raise RuntimeError('Expected the two workshop sample documents under data/search.')
for path in sample_files:
    blob_name = f'{blob_prefix}{path.name}'
    response = requests.put(
        f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
        headers={
            'Authorization': f'Bearer {storage_token()}',
            'x-ms-version': '2023-11-03',
            'x-ms-date': format_datetime(datetime.now(timezone.utc), usegmt=True),
            'x-ms-blob-type': 'BlockBlob',
            'Content-Type': 'text/plain; charset=utf-8',
        },
        data=path.read_bytes(), timeout=(10, 90),
    )
    response.raise_for_status()
    print('Uploaded:', blob_name)


## 3. Create the Blob data source

`ResourceId=...;` tells Search to authenticate to Storage using its system-assigned managed identity. `container.query` scopes the indexer to your virtual folder, so it does not ingest everyone else's workshop documents. This is a data selection rule, not a security boundary.

We deliberately omit a deletion policy in this introductory lab. Deleting a source blob alone does not promise removal of its indexed chunks. Configure and test a supported deletion-detection policy before using this pattern for content that must be removed automatically.

In [ ]:
data_source = {
    'name': data_source_name,
    'type': 'azureblob',
    'credentials': {'connectionString': f'ResourceId={storage_id};'},
    'container': {'name': container_name, 'query': blob_prefix},
}
response = requests.put(
    f'{search_endpoint}/datasources/{data_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=data_source, timeout=(10, 90),
)
response.raise_for_status()
print('Data source ready:', data_source_name)


## 4. Define a chunk index

Each chunk will be a Search document. `chunk_id` is generated by index projections; `parent_id` groups chunks from one source file and must be filterable. Do not manually map either field in the projection. The complex `source` field holds provenance produced by Shaper.

In [ ]:
index = {
    'name': index_name,
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'source', 'type': 'Edm.ComplexType', 'fields': [
            {'name': 'file_name', 'type': 'Edm.String', 'searchable': True},
            {'name': 'url', 'type': 'Edm.String'},
        ]},
    ],
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=index, timeout=(10, 90),
)
response.raise_for_status()
print('Index ready:', index_name)


## 5. Define Text Split and Shaper skills

The indexer first cracks each source file into `/document/content` and metadata. Text Split writes strings to `/document/chunks/*`; Shaper writes a complex object to `/document/provenance`. Index projections repeat that object on every chunk.

`textSplitMode='pages'` means logical chunks, **not physical PDF pages**. We use 1,000 characters with 200-character overlap. These settings are small so you can inspect chunk boundaries in the workshop. They do not provide page-number citations.

In [ ]:
skillset = {
    'name': skillset_name,
    'description': 'Split blob text and preserve structured source provenance.',
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Text.SplitSkill',
            'name': 'split-text', 'context': '/document',
            'textSplitMode': 'pages', 'maximumPageLength': 1000, 'pageOverlapLength': 200,
            'inputs': [{'name': 'text', 'source': '/document/content'}],
            'outputs': [{'name': 'textItems', 'targetName': 'chunks'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Util.ShaperSkill',
            'name': 'shape-provenance', 'context': '/document',
            'inputs': [
                {'name': 'file_name', 'source': '/document/metadata_storage_name'},
                {'name': 'url', 'source': '/document/metadata_storage_path'},
            ],
            'outputs': [{'name': 'output', 'targetName': 'provenance'}],
        },
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/chunks/*',
            'mappings': [
                {'name': 'chunk', 'source': '/document/chunks/*'},
                {'name': 'source', 'source': '/document/provenance'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
response = requests.put(
    f'{search_endpoint}/skillsets/{skillset_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=skillset, timeout=(10, 90),
)
response.raise_for_status()
print('Skillset ready:', skillset_name)


## 6. Create the indexer

The indexer connects the data source, skillset and target index. Creating it starts its first run automatically. When this cell updates an existing idle indexer, it explicitly requests a new run; an already-running indexer must finish first. The first run can take several minutes while permissions propagate. No schedule is configured, so additional runs are explicit.

`maxFailedItems=0` and `maxFailedItemsPerBatch=0` prevent a partially failed run from being presented as a successful workshop outcome.

In [ ]:
response = requests.get(
    f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
indexer_exists = response.status_code != 404
previous_run_start = None
if indexer_exists:
    response.raise_for_status()
    previous_run = response.json().get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {'dataToExtract': 'contentAndMetadata'},
    },
}
response = requests.put(
    f'{search_endpoint}/indexers/{indexer_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=indexer, timeout=(10, 90),
)
response.raise_for_status()
print('Indexer ready:', indexer_name)
if indexer_exists:
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('New run requested for the existing indexer.')


In [ ]:
for attempt in range(90):
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    status = response.json()
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')


## 7. Inspect the results and run a keyword query

First inspect all chunks and their provenance. More chunks than source files demonstrates one-to-many projection. Then search for the synthetic travel policy. This is full-text search, not vector or semantic search; no embedding model is required.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'count': True, 'select': 'chunk_id,parent_id,chunk,source', 'top': 100},
    timeout=(10, 90),
)
response.raise_for_status()
results = response.json()
chunks = results['value']
if not chunks:
    raise RuntimeError('No chunks were indexed. Inspect indexer errors and the Blob folder prefix.')
print('Total chunks:', results['@odata.count'])
print(json.dumps(chunks, indent=2))


In [ ]:
question = 'travel reimbursement receipts'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': question, 'searchFields': 'chunk', 'select': 'chunk,source', 'top': 3},
    timeout=(10, 90),
)
response.raise_for_status()
print(json.dumps(response.json(), indent=2))


## 8. Experiments: incremental indexing and chunk size

1. Add a text-based document to your Blob folder, run the following cell, then rerun the monitor and inspection cells. Creating an indexer is not required again. Changed/new blobs are picked up incrementally.
2. Change `maximumPageLength` to 2,000 and overlap to 300 in the skillset, rerun that cell, then enable the reset in the next cell before running. Updating a skillset alone does not reprocess unchanged blobs. Compare chunk counts and retrieval results.
3. Explain why the returned Blob URL still requires authenticated access and why a logical chunk is not a page citation.

The run endpoint returns a conflict if the indexer is already running. Wait for that run to finish rather than hiding the error. For a changed chunking strategy, rebuilding your own lab index is also an option; never reset or delete a classmate's objects.

In [ ]:
RUN_AGAIN = False
RESET_BEFORE_RUN = False
if RUN_AGAIN:
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    last_run = response.json().get('lastResult') or {}
    if last_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait for it to finish.')
    previous_run_start = last_run.get('startTime')
    if RESET_BEFORE_RUN:
        response = requests.post(
            f'{search_endpoint}/indexers/{indexer_name}/reset?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        response.raise_for_status()
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('Run requested. Rerun the monitor and result cells.')
else:
    print('No extra run requested. Set RUN_AGAIN=True for the exercise.')


## 9. Optional cleanup

Opt in to delete only this notebook's four named Search objects and its two uploaded sample blobs. It does not delete the Search service, container, storage account, other files you added, or resources created by other labs. Blob soft deletion can retain billable copies.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [('indexers', indexer_name), ('skillsets', skillset_name),
                             ('indexes', index_name), ('datasources', data_source_name)]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
    for path in sample_files:
        blob_name = f'{blob_prefix}{path.name}'
        response = requests.delete(
            f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
            headers={'Authorization': f'Bearer {storage_token()}', 'x-ms-version': '2023-11-03'},
            timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', blob_name)
else:
    print('Cleanup disabled.')


## References

- [Blob indexer](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-storage)
- [Managed identity access to Storage](https://learn.microsoft.com/azure/search/search-how-to-managed-identities-storage)
- [Text Split skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit)
- [Shaper skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-shaper)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Blob deletion detection](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-changed-deleted)

Next: compare SharePoint indexed ingestion with remote retrieval in notebook 04. Document Intelligence Layout, Content Understanding and text embeddings can be introduced in later labs without adding a Vision skill.